In [4]:
import pandas as pd
from sqlalchemy import select

from models.app_models import Magasin
from utils.connections import app_session_factory

with app_session_factory() as app_session:
    query = select(Magasin).where(Magasin.actif)
    df = pd.read_sql(query, app_session.connection())

df.head()
    

,magasin_id,nom,adresse,ville,pays,date_ouverture,actif,code_postal
0,1,Givre & Co - Grand-Place,Rue du Marché aux Herbes 25,Bruxelles,Belgique,2019-03-15,True,1000
1,2,Givre & Co - Carré,Place du Marché 15,LIEGE,Belgique,2020-04-01,True,4000
2,3,Givre & Co - Namur,Rue de Fer 30,Namur,Belgique,2021-05-01,True,NaN
3,4,Givre & Co - Gent,Veldstraat 50,Gent,Belgique,2022-03-20,True,9000
4,5,Givre & Co - Digue,Albert I-Promenade 40,Oostende,Belgique,2018-04-01,True,8400


In [5]:
import asyncio

from utils.api import get_address_infos

# normalisation de la colonne ville
df['ville'] = df['ville'].str.capitalize()

for i, line in enumerate(df.itertuples()):
    infos = await get_address_infos(line.adresse, line.ville)

    # modifier le code postal car il est parfois manquant
    df.loc[i, 'code_postal'] = str(infos.get('address').get('postcode'))
    
    # ajout des nouvelles colonnes
    df.loc[i, 'latitude'] = float(infos.get('lat'))
    df.loc[i, 'longitude'] = float(infos.get('lon'))
    df.loc[i, 'region'] = infos.get('address').get('state') or infos.get('address').get('county')
    df.loc[i, 'est_cotier'] = float(infos.get('lon')) < 3.35 

    # l'api nominatim est un peu chiante 
    # on doit attendre un peu pour éviter d'être blacklisté
    await asyncio.sleep(2)

df.head()

,magasin_id,nom,adresse,ville,pays,date_ouverture,actif,code_postal,latitude,longitude,region,est_cotier
0,1,Givre & Co - Grand-Place,Rue du Marché aux Herbes 25,Bruxelles,Belgique,2019-03-15,True,1000,50.848065,4.352270,Brussel-Hoofdstad - Bruxelles-Capitale,False
1,2,Givre & Co - Carré,Place du Marché 15,Liege,Belgique,2020-04-01,True,4600,50.733828,5.697437,Liège,False
2,3,Givre & Co - Namur,Rue de Fer 30,Namur,Belgique,2021-05-01,True,5000,50.465762,4.865540,Namur,False
3,4,Givre & Co - Gent,Veldstraat 50,Gent,Belgique,2022-03-20,True,9890,50.919359,3.645297,Oost-Vlaanderen,False
4,5,Givre & Co - Digue,Albert I-Promenade 40,Oostende,Belgique,2018-04-01,True,8400,51.234280,2.915651,West-Vlaanderen,True


In [6]:
from models.dw_models import DimMagasin
from utils.connections import dw_session_factory

with dw_session_factory() as dw_session:
    df.drop(columns=['actif', 'pays']).to_sql(
        DimMagasin.__tablename__, 
        if_exists='append', 
        index=False, 
        con=dw_session.connection(),
    )
    dw_session.commit()